In [ ]:
from __future__ import annotations

import mlflow
import mlflow.onnx
import pandas as pd
from lightgbm import LGBMClassifier
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient
from onnxmltools.convert.lightgbm.operator_converters.LightGbm import convert_lightgbm
from skl2onnx import update_registered_converter
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

from flip_flopper.classifier_training import (
    TARGET_COLUMN,
    evaluate_holdout,
    feature_columns_from_frame,
    load_training_frame,
    prepare_numeric_features,
    qualified_table_name,
    resolve_effective_cv_folds,
    resolve_experiment_path,
    summarize_cv_metrics,
    uc_registered_model_name,
)
from flip_flopper.sklearn_onnx import (
    DEFAULT_TARGET_OPSET,
    build_per_column_onnx_input_example,
    convert_pipeline_to_onnx,
    validate_onnx_probabilities,
)

update_registered_converter(
    LGBMClassifier,
    "LightGbmLGBMClassifier",
    calculate_linear_classifier_output_shapes,
    convert_lightgbm,
    options={"nocl": [True, False], "zipmap": [True, False]},
)

RANDOM_STATE = 42
MAX_TRAINING_ROWS_DEFAULT = 200_000
CV_FOLDS_DEFAULT = 5
TEST_SIZE = 0.2
N_ESTIMATORS = 300
LIGHTGBM_TARGET_OPSET = {"": DEFAULT_TARGET_OPSET, "ai.onnx.ml": 3}
MODEL_ALIAS = "Champion"
MODEL_VARIANT = "l"


# Job parameters are passed in through notebook task base_parameters.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "flip_flopper")
dbutils.widgets.text("table_name", "generated_data")
dbutils.widgets.text("experiment_name", "flip_flopper_classifier")
dbutils.widgets.text("registered_model_suffix", "flip_flopper_classifier")
dbutils.widgets.text("max_training_rows", str(MAX_TRAINING_ROWS_DEFAULT))
dbutils.widgets.text("cv_folds", str(CV_FOLDS_DEFAULT))


config = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "table_name": dbutils.widgets.get("table_name").strip(),
    "experiment_name": dbutils.widgets.get("experiment_name").strip(),
    "registered_model_suffix": dbutils.widgets.get("registered_model_suffix").strip(),
    "max_training_rows": int(dbutils.widgets.get("max_training_rows").strip()),
    "cv_folds": int(dbutils.widgets.get("cv_folds").strip()),
}


def build_training_pipeline(feature_columns: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                feature_columns,
            )
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    classifier = LGBMClassifier(
        n_estimators=N_ESTIMATORS,
        class_weight="balanced",
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=-1,
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("classifier", classifier),
        ]
    )


source_table = qualified_table_name(
    config["catalog"], config["schema"], config["table_name"]
)
experiment_path = resolve_experiment_path(config["experiment_name"])
registered_model_name = uc_registered_model_name(
    config["catalog"],
    config["schema"],
    config["registered_model_suffix"],
    MODEL_VARIANT,
)

training_frame = load_training_frame(
    spark,
    source_table,
    max_training_rows=config["max_training_rows"],
    random_state=RANDOM_STATE,
)
training_frame[TARGET_COLUMN] = pd.to_numeric(
    training_frame[TARGET_COLUMN], errors="raise"
).astype("int64")

feature_columns = feature_columns_from_frame(training_frame)
features, labels = prepare_numeric_features(training_frame, feature_columns)

x_train, x_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=RANDOM_STATE,
)

training_pipeline = build_training_pipeline(feature_columns)

requested_cv_folds = config["cv_folds"]
effective_cv_folds = resolve_effective_cv_folds(requested_cv_folds, y_train)

cv = StratifiedKFold(
    n_splits=effective_cv_folds, shuffle=True, random_state=RANDOM_STATE
)
cv_results = cross_validate(
    training_pipeline,
    x_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc",
    },
    n_jobs=1,
    return_train_score=False,
)

training_pipeline.fit(x_train, y_train)
holdout_metrics = evaluate_holdout(training_pipeline, x_test, y_test)
cross_validation_metrics = summarize_cv_metrics(cv_results)

onnx_model = convert_pipeline_to_onnx(
    training_pipeline,
    feature_columns,
    target_opset=LIGHTGBM_TARGET_OPSET,
)
validation_batch = x_test.head(min(128, len(x_test))).copy()
onnx_input_example = build_per_column_onnx_input_example(
    validation_batch, feature_columns
)

onnx_probability_max_abs_diff, onnx_proba_2d = validate_onnx_probabilities(
    training_pipeline, onnx_model, validation_batch, feature_columns
)

signature = infer_signature(validation_batch, onnx_proba_2d)

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

with mlflow.start_run(run_name="train_classifier_lightgbm") as _:
    mlflow.log_params(
        {
            "algorithm": "lightgbm_classifier",
            "catalog": config["catalog"],
            "schema": config["schema"],
            "source_table": f"{config['catalog']}.{config['schema']}.{config['table_name']}",
            "registered_model_name": registered_model_name,
            "feature_count": len(feature_columns),
            "training_row_count": len(x_train),
            "test_row_count": len(x_test),
            "max_training_rows": config["max_training_rows"],
            "cv_folds_requested": requested_cv_folds,
            "cv_folds_effective": effective_cv_folds,
            "n_estimators": N_ESTIMATORS,
        }
    )
    mlflow.log_metrics(cross_validation_metrics)
    mlflow.log_metrics(holdout_metrics)
    mlflow.log_metric("onnx_probability_max_abs_diff", onnx_probability_max_abs_diff)
    mlflow.log_table(pd.DataFrame(cv_results), artifact_file="cv_results.json")
    mlflow.log_text("\n".join(feature_columns), artifact_file="feature_columns.txt")

    model_info = mlflow.onnx.log_model(
        onnx_model=onnx_model,
        name="model",
        input_example=onnx_input_example,
        signature=signature,
        registered_model_name=registered_model_name,
        metadata={
            "algorithm": "lightgbm_classifier",
            "source_table": f"{config['catalog']}.{config['schema']}.{config['table_name']}",
            "notebook": "train_classifier_lightgbm",
        },
    )
    MlflowClient().set_registered_model_alias(
        name=registered_model_name,
        alias=MODEL_ALIAS,
        version=model_info.registered_model_version,
    )

print(f"Experiment path: {experiment_path}")
print(f"Registered model: {registered_model_name}")
print(f"Registered model alias: {MODEL_ALIAS}")
print(f"Model URI: {model_info.model_uri}")
print(f"Cross-validation metrics: {cross_validation_metrics}")
print(f"Holdout metrics: {holdout_metrics}")
print(f"ONNX probability max abs diff: {onnx_probability_max_abs_diff:.8f}")